3.1 Imports

In [1]:
import os
import pandas as pd
import numpy as np
from itables import init_notebook_mode, show


os.chdir('C:/Users/lhcse/iCloudDrive/Learnings/Power-price-forecast')

3.2 Load raw files

In [2]:
# Load all six files
prices      = pd.read_csv('data/raw/day_ahead_prices_smard.csv',
                           sep=';', thousands=',', na_values='-')
gen_actual  = pd.read_csv('data/raw/generation_actual_smard.csv',
                           sep=';', thousands=',', na_values='-')
gen_fc      = pd.read_csv('data/raw/generation_forecast_smard.csv',
                           sep=';', thousands=',', na_values='-')
load_actual = pd.read_csv('data/raw/consumption_actual_smard.csv',
                           sep=';', thousands=',', na_values='-')
load_fc     = pd.read_csv('data/raw/consumption_forecasts_smard.csv',
                           sep=';', thousands=',', na_values='-')
crossborder = pd.read_csv('data/raw/crossborder_smard.csv',
                           sep=';', thousands=',', na_values='-')

#3.3 Inspect each file

In [3]:
for name, df in [
    ('prices',        prices),
    ('gen_actual',    gen_actual),
    ('gen_forecast',  gen_fc),
    ('load_actual',   load_actual),
    ('load_forecast', load_fc),
    ('crossborder',   crossborder)
]:
    print(f"\n── {name} ──")
    print(f"Shape:   {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print(df.head(2))


── prices ──
Shape:   (43848, 19)
Columns: ['Start date', 'End date', 'Germany/Luxembourg [€/MWh] Calculated resolutions', '∅ DE/LU neighbours [€/MWh] Calculated resolutions', 'Belgium [€/MWh] Calculated resolutions', 'Denmark 1 [€/MWh] Calculated resolutions', 'Denmark 2 [€/MWh] Calculated resolutions', 'France [€/MWh] Calculated resolutions', 'Netherlands [€/MWh] Calculated resolutions', 'Norway 2 [€/MWh] Calculated resolutions', 'Austria [€/MWh] Calculated resolutions', 'Poland [€/MWh] Calculated resolutions', 'Sweden 4 [€/MWh] Calculated resolutions', 'Switzerland [€/MWh] Calculated resolutions', 'Czech Republic [€/MWh] Calculated resolutions', 'DE/AT/LU [€/MWh] Calculated resolutions', 'Northern Italy [€/MWh] Calculated resolutions', 'Slovenia [€/MWh] Calculated resolutions', 'Hungary [€/MWh] Calculated resolutions']
             Start date             End date  \
0  Jan 1, 2020 12:00 AM  Jan 1, 2020 1:00 AM   
1   Jan 1, 2020 1:00 AM  Jan 1, 2020 2:00 AM   

   Germany/Luxembour

In [4]:
from itables import show
show(prices)

Loading ITables v2.8.0 from the internet... (need help?)


#3.4 Parse timestamps

In [5]:
from itables import show
show(gen_actual)

Loading ITables v2.8.0 from the internet... (need help?)


In [6]:
def parse_smard_time(df, time_col='Start date'):
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col], dayfirst=True)
    df = df.set_index(time_col)
    df = df.sort_index()
    return df

prices      = parse_smard_time(prices)
gen_actual  = parse_smard_time(gen_actual)
gen_fc      = parse_smard_time(gen_fc)
load_actual = parse_smard_time(load_actual)
load_fc     = parse_smard_time(load_fc)
crossborder = parse_smard_time(crossborder)

C:\Users\lhcse\AppData\Local\Temp\ipykernel_12816\1117611002.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], dayfirst=True)
C:\Users\lhcse\AppData\Local\Temp\ipykernel_12816\1117611002.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], dayfirst=True)
C:\Users\lhcse\AppData\Local\Temp\ipykernel_12816\1117611002.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[time_col] = pd.to_datetime(df[time_col], dayfirst=True)
C:\Users\lhcse\AppData\Local\Temp\ipykernel_12816\1117611002.py:

#3.5 Check for missing values

In [7]:
for name, df in [
    ('prices',        prices),
    ('gen_actual',    gen_actual),
    ('gen_forecast',  gen_fc),
    ('load_actual',   load_actual),
    ('load_forecast', load_fc),
    ('crossborder',   crossborder)
]:
    missing = df.isnull().sum().sum()
    print(f"{name:20s}  rows: {len(df):6,}  missing values: {missing:,}")

prices                rows: 43,848  missing values: 43,848
gen_actual            rows: 43,848  missing values: 8,076
gen_forecast          rows: 43,848  missing values: 0
load_actual           rows: 43,848  missing values: 0
load_forecast         rows: 43,848  missing values: 48
crossborder           rows: 43,848  missing values: 31,075


#3.6 Align and merge into one dataframe

In [8]:
# ── Step 1: Fix price column and drop unnecessary columns ─────────────────
prices = prices[['End date', 'Germany/Luxembourg [€/MWh] Calculated resolutions']].copy()
prices = prices.rename(columns={
    'Germany/Luxembourg [€/MWh] Calculated resolutions': 'price_eur_mwh'
})
prices = prices.drop(columns=['End date'])

# ── Step 2: Drop End date from all other files ────────────────────────────
for name in ['gen_actual', 'gen_fc', 'load_actual', 'load_fc', 'crossborder']:
    globals()[name].drop(columns=['End date'], inplace=True, errors='ignore')

# ── Step 3: Add suffixes once ─────────────────────────────────────────────
gen_actual  = gen_actual.add_suffix('_gen_act')
gen_fc      = gen_fc.add_suffix('_gen_fc')
load_actual = load_actual.add_suffix('_load_act')
load_fc     = load_fc.add_suffix('_load_fc')
crossborder = crossborder.add_suffix('_cross')

# ── Step 4: Merge ──────────────────────────────────────────────────────────
df = prices[['price_eur_mwh']].copy()
df = df.join(gen_actual,  how='inner')
df = df.join(gen_fc,      how='inner')
df = df.join(load_actual, how='inner')
df = df.join(load_fc,     how='inner')
df = df.join(crossborder, how='inner')

# ── Step 5: Remove duplicate timestamps (from DST clock changes) ──────────
n_before = len(df)
df = df[~df.index.duplicated(keep='first')]
print(f"Removed {n_before - len(df)} duplicate timestamps (DST)")

print(f"\nMerged shape: {df.shape}")
print(f"Date range:   {df.index.min()} to {df.index.max()}")
print(f"\nColumns:\n{list(df.columns)}")

df.to_parquet('data/processed/merged_raw.parquet', index=True)

Removed 315 duplicate timestamps (DST)

Merged shape: (43843, 48)
Date range:   2020-01-01 00:00:00 to 2024-12-31 23:00:00

Columns:
['price_eur_mwh', 'Biomass [MWh] Calculated resolutions_gen_act', 'Hydropower [MWh] Calculated resolutions_gen_act', 'Wind offshore [MWh] Calculated resolutions_gen_act', 'Wind onshore [MWh] Calculated resolutions_gen_act', 'Photovoltaics [MWh] Calculated resolutions_gen_act', 'Other renewable [MWh] Calculated resolutions_gen_act', 'Nuclear [MWh] Calculated resolutions_gen_act', 'Lignite [MWh] Calculated resolutions_gen_act', 'Hard coal [MWh] Calculated resolutions_gen_act', 'Fossil gas [MWh] Calculated resolutions_gen_act', 'Hydro pumped storage [MWh] Calculated resolutions_gen_act', 'Other conventional [MWh] Calculated resolutions_gen_act', 'Total [MWh] Calculated resolutions_gen_fc', 'Photovoltaics and wind [MWh] Calculated resolutions_gen_fc', 'Wind offshore [MWh] Calculated resolutions_gen_fc', 'Wind onshore [MWh] Calculated resolutions_gen_fc', 'Pho

In [9]:
rename_map = {
    # Generation actual
    'Biomass [MWh] Calculated resolutions_gen_act':              'biomass_gen_act',
    'Hydropower [MWh] Calculated resolutions_gen_act':           'hydro_gen_act',
    'Wind offshore [MWh] Calculated resolutions_gen_act':        'wind_off_gen_act',
    'Wind onshore [MWh] Calculated resolutions_gen_act':         'wind_on_gen_act',
    'Photovoltaics [MWh] Calculated resolutions_gen_act':        'solar_gen_act',
    'Other renewable [MWh] Calculated resolutions_gen_act':      'other_ren_gen_act',
    'Nuclear [MWh] Calculated resolutions_gen_act':              'nuclear_gen_act',
    'Lignite [MWh] Calculated resolutions_gen_act':              'lignite_gen_act',
    'Hard coal [MWh] Calculated resolutions_gen_act':            'hardcoal_gen_act',
    'Fossil gas [MWh] Calculated resolutions_gen_act':           'gas_gen_act',
    'Hydro pumped storage [MWh] Calculated resolutions_gen_act': 'hydro_pump_gen_act',
    'Other conventional [MWh] Calculated resolutions_gen_act':   'other_conv_gen_act',

    # Generation forecast
    'Total [MWh] Calculated resolutions_gen_fc':                 'total_gen_fc',
    'Photovoltaics and wind [MWh] Calculated resolutions_gen_fc':'solar_wind_gen_fc',
    'Wind offshore [MWh] Calculated resolutions_gen_fc':         'wind_off_gen_fc',
    'Wind onshore [MWh] Calculated resolutions_gen_fc':          'wind_on_gen_fc',
    'Photovoltaics [MWh] Calculated resolutions_gen_fc':         'solar_gen_fc',
    'Other [MWh] Calculated resolutions_gen_fc':                 'other_gen_fc',

    # Load actual
    'grid load [MWh] Calculated resolutions_load_act':                              'load_act',
    'Grid load incl. hydro pumped storage [MWh] Calculated resolutions_load_act':   'load_incl_pump_act',
    'Hydro pumped storage [MWh] Calculated resolutions_load_act':                   'hydro_pump_load_act',
    'Residual load [MWh] Calculated resolutions_load_act':                          'residual_load_act',

    # Load forecast
    'grid load [MWh] Calculated resolutions_load_fc':            'load_fc',
    'Residual load [MWh] Calculated resolutions_load_fc':        'residual_load_fc',

    # Cross-border
    'Net export [MWh] Calculated resolutions_cross':             'net_export',
    'Netherlands (export) [MWh] Calculated resolutions_cross':   'nl_export',
    'Netherlands (import) [MWh] Calculated resolutions_cross':   'nl_import',
    'Switzerland (export) [MWh] Calculated resolutions_cross':   'ch_export',
    'Switzerland (import) [MWh] Calculated resolutions_cross':   'ch_import',
    'Denmark (export) [MWh] Calculated resolutions_cross':       'dk_export',
    'Denmark (import) [MWh] Calculated resolutions_cross':       'dk_import',
    'Czech Republic (export) [MWh] Calculated resolutions_cross':'cz_export',
    'Czech Republic (import) [MWh] Calculated resolutions_cross':'cz_import',
    'Luxembourg (export) [MWh] Calculated resolutions_cross':    'lu_export',
    'Luxembourg (import) [MWh] Calculated resolutions_cross':    'lu_import',
    'Sweden (export) [MWh] Calculated resolutions_cross':        'se_export',
    'Sweden (import) [MWh] Calculated resolutions_cross':        'se_import',
    'Austria (export) [MWh] Calculated resolutions_cross':       'at_export',
    'Austria (import) [MWh] Calculated resolutions_cross':       'at_import',
    'France (export) [MWh] Calculated resolutions_cross':        'fr_export',
    'France (import) [MWh] Calculated resolutions_cross':        'fr_import',
    'Poland (export) [MWh] Calculated resolutions_cross':        'pl_export',
    'Poland (import) [MWh] Calculated resolutions_cross':        'pl_import',
    'Norway (export) [MWh] Calculated resolutions_cross':        'no_export',
    'Norway (import) [MWh] Calculated resolutions_cross':        'no_import',
    'Belgium (export) [MWh] Calculated resolutions_cross':       'be_export',
    'Belgium (import) [MWh] Calculated resolutions_cross':       'be_import',
}

df = df.rename(columns=rename_map)
df.to_parquet('data/processed/merged_raw.parquet')

print("Column names after renaming:")
print(list(df.columns))

Column names after renaming:
['price_eur_mwh', 'biomass_gen_act', 'hydro_gen_act', 'wind_off_gen_act', 'wind_on_gen_act', 'solar_gen_act', 'other_ren_gen_act', 'nuclear_gen_act', 'lignite_gen_act', 'hardcoal_gen_act', 'gas_gen_act', 'hydro_pump_gen_act', 'other_conv_gen_act', 'total_gen_fc', 'solar_wind_gen_fc', 'wind_off_gen_fc', 'wind_on_gen_fc', 'solar_gen_fc', 'other_gen_fc', 'load_act', 'load_incl_pump_act', 'hydro_pump_load_act', 'residual_load_act', 'load_fc', 'residual_load_fc', 'net_export', 'nl_export', 'nl_import', 'ch_export', 'ch_import', 'dk_export', 'dk_import', 'cz_export', 'cz_import', 'lu_export', 'lu_import', 'se_export', 'se_import', 'at_export', 'at_import', 'fr_export', 'fr_import', 'pl_export', 'pl_import', 'no_export', 'no_import', 'be_export', 'be_import']
